# NASA Turbofan 엔진 잔여수명(RUL) 예측 — 실습

> **세션 · 항공 예지 보전 심층 분석**  |  데이터: *NASA C-MAPSS Turbofan Engine Degradation* (NASA PCoE)

---

## 0. 이 노트북의 목표

> **"지금 이 엔진, 앞으로 몇 사이클을 더 돌고 고장날까? — 센서 데이터만으로 맞힐 수 있을까?"**

항공 엔진이 비행 중 고장나면 인명 손실로 이어집니다. 반대로 너무 일찍 교체하면 거대한 비용 낭비.
**잔여수명(RUL, Remaining Useful Life) 예측**은 그 사이 균형을 잡는 예지 보전의 핵심 과제입니다.

이 노트북의 진짜 주제는 *"낮은 RMSE 자랑"*이 아니라 **"정직한 평가"** 입니다. 실제로 우리는
**"제대로 평가하면 흔히 보는 baseline 숫자가 과대평가되어 있다"** 는 결론에 도달하고, 그것이 왜
시퀀스 모델·도메인 적응이 필요한지를 보여줍니다.

### 산출물 스펙 매핑

| 스펙 항목 | 다루는 섹션 |
|---|---|
| **1-1 핵심 알고리즘** | 4(Piecewise RUL · Rolling FE) · 6(RandomForest · GBR) · 7(LSTM · Ridge) |
| **1-2 EDA(데이터 특성)** | 3. EDA |
| **1-3 왜 이렇게 푸는가** | 5. ⭐ 그룹 누수·시간 누수·유효표본 |
| **1-4 코드별 설명** | 전 코드 셀 하단 **[코드 설명]/[해석]** |

### 참조 Kaggle 메달 노트북 (Top 3)
| 티어 | 노트북 | 핵심 |
|---|---|---|
| 🥇 Gold | [wassimderbel — Nasa predictive Maintenance (RUL)](https://www.kaggle.com/code/wassimderbel/nasa-predictive-maintenance-rul) | RF / XGBoost / LSTM 비교 |
| 🥈 Silver | [carlkirstein — Predictive Maintenance NASA turbofan Regression](https://www.kaggle.com/code/carlkirstein/predictive-maintenance-nasa-turbofan-regression) | Rolling FE + 회귀 |
| 🥉 Bronze | [mikenguyen1712 — Predict RUL using LSTM and AR](https://www.kaggle.com/code/mikenguyen1712/nasa-turbofan-predict-rul-using-lstm-and-ar-model) | LSTM + AR 하이브리드 |

---


## 1. 도메인 이해 — Turbofan 엔진과 RUL

### 1.1 Turbofan 엔진
민간 항공기 주력 엔진. 핵심 작동: **공기 흡입 → 압축 → 연료 분사·연소 → 팽창 분사 (추력)**.
- **Fan**: 거대한 팬이 공기를 흡입
- **Compressor (LPC/HPC)**: 저압·고압 압축기
- **Combustor**: 연소실
- **Turbine (HPT/LPT)**: 고압·저압 터빈 — 압축기를 돌리는 회전 동력 회수
- **Nozzle**: 배기 노즐

### 1.2 엔진 열화(degradation)가 왜 중요한가
운용 사이클이 누적되면 **HPC 효율 저하·Fan blade 미세 손상** 이 진행됩니다.
- 같은 출력에 **더 큰 연료·온도** 필요
- 센서 값(EGT, 압력비, 회전수, 연료 유량)에 **trend** 로 나타남

→ 이 trend가 **얼마나 진행됐고 앞으로 몇 사이클 남았나** 가 RUL.

### 1.3 데이터셋: NASA C-MAPSS
NASA Ames의 시뮬레이션 환경에서 100~250대 가상 엔진을 **고장날 때까지** 운영해 21개 센서를 기록.
4개 서브셋 — FD001(쉬움) ~ FD004(가장 어려움).

| Subset | Train 엔진 | 운영조건 | 고장모드 | 난이도 |
|---|---:|---:|---:|---|
| **FD001** | **100** | 1 | 1 (HPC) | ⭐ 본 노트북 대상 |
| FD002 | 260 | 6 | 1 | ⭐⭐⭐ |
| FD003 | 100 | 1 | 2 | ⭐⭐ |
| FD004 | 249 | 6 | 2 | ⭐⭐⭐⭐ |

> **⚠️ 중요 — 이 데이터는 'run-to-failure 시뮬레이션'**
> 실제 비행 데이터가 아니라 C-MAPSS 시뮬레이션 출력입니다. 노이즈 모델·열화 곡선이 인공적이므로
> 실기체 적용 시엔 도메인 적응이 필요합니다. 다만 **알고리즘 벤치마크용 표준**으로 20년간 자리잡았습니다.

### 1.4 컬럼 구조

**`train_FD001.txt` (20,631행 — 100엔진 × ~206 cycle 평균)**

| 컬럼 | 의미 | 역할 |
|---|---|---|
| `unit` | 엔진 ID 1~100 | 식별자 → **⭐ 그룹 키** |
| `cycle` | 누적 운영 사이클 | 시간 축 |
| `op_setting_1~3` | 비행 조건 (고도·Mach·Throttle) | 입력 조건 |
| `sensor_1~21` | 온도·압력·회전수·연료비 등 | 입력 특징 |
| **(계산된) `RUL`** | **남은 cycle 수** | **⭐ 예측 대상** |

**`test_FD001.txt` (13,096행)**: 동일 구조지만 **고장 전 임의 시점에서 잘림**.
**`RUL_FD001.txt` (100행)**: 각 test 엔진의 **마지막 cycle에 대한 정답 RUL**.

---


## 2. 환경 설정 & 데이터 로드  `[1-4 코드별 설명]`

> **💻 실행 환경** — **Colab / 로컬 모두 지원**. 아래 첫 셀이 환경을 감지해 (Colab이면) 패키지·한글폰트를 설치합니다.


In [ ]:
# --- [환경 설정] Colab 자동 감지 → 패키지 & 한글 폰트 설치 ---
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run("pip install -q lightgbm", shell=True)
    subprocess.run("apt-get -qq -y install fonts-nanum > /dev/null 2>&1", shell=True)
    import matplotlib.font_manager as fm
    for p in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf"]:
        try: fm.fontManager.addfont(p)
        except Exception: pass
    print("Colab 환경 설정 완료")
else:
    print("로컬 환경")

# --- 라이브러리 ---
import os, glob, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["axes.unicode_minus"] = False
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

import matplotlib.font_manager as _fm
_avail = {f.name for f in _fm.fontManager.ttflist}
for _f in ["AppleGothic", "NanumGothic", "Malgun Gothic"]:
    if _f in _avail:
        plt.rcParams["font.family"] = _f; print("한글 폰트:", _f); break
else:
    print("⚠️ 한글 폰트 미발견 — 그래프 한글이 깨질 수 있음")


### 📥 데이터 준비
로컬은 `data/cmapss/` 폴더에 NASA PCoE의 4개 서브셋(train/test/RUL × FD001~FD004) 텍스트 파일을 두면 됩니다.
**Colab**이면 아래 중 하나로 올린 뒤 진행하세요.

```python
# 방법 A) Kaggle API (kaggle.json 업로드)
from google.colab import files; files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d behrad3d/nasa-cmaps -p data/cmapss --unzip
# 방법 B) zip 직접 업로드 → !mkdir -p data/cmapss && unzip -q '*.zip' -d data/cmapss
```


In [ ]:
!mkdir -p data/cmapss && unzip -q '*.zip' -d data/cmapss

In [ ]:
def find_data_dir():
    for base in ["data/cmapss", "data", ".", "/kaggle/input/nasa-cmaps"]:
        hits = glob.glob(os.path.join(base, "**", "train_FD001.txt"), recursive=True)
        if hits: return os.path.dirname(hits[0])
    raise FileNotFoundError("train_FD001.txt 없음 → data/cmapss/ 폴더에 데이터셋을 넣어주세요.")
DATA_DIR = find_data_dir(); print("데이터 경로:", DATA_DIR)

# --- 컬럼명 정의 ---
COL_NAMES = (
    ["unit", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}"     for i in range(1, 22)]
)

def load_subset(name):
    train  = pd.read_csv(f"{DATA_DIR}/train_{name}.txt", sep=r"\s+", header=None, names=COL_NAMES)
    test   = pd.read_csv(f"{DATA_DIR}/test_{name}.txt",  sep=r"\s+", header=None, names=COL_NAMES)
    y_test = pd.read_csv(f"{DATA_DIR}/RUL_{name}.txt",   sep=r"\s+", header=None, names=["RUL"])
    return train, test, y_test

train, test, y_test = load_subset("FD001")
print(f"train: {train.shape} (행/열)")
print(f"test : {test.shape}")
print(f"정답 : {y_test.shape}  ← 엔진당 1개씩, 100엔진 = 100개")
train.head(3)


**[코드 설명]** `sep=r"\s+"` — 이 데이터는 공백 여러 개로 구분되는 비표준 포맷. 정규표현식 `\s+` 로 처리.
`load_subset()` 은 FD001~FD004 4개 서브셋에 대해 동일 작업을 재사용할 수 있게 함수로 묶었습니다.

**[관찰]**
- train: **20,631행 × 26열** — 엔진 100대 × 평균 206 cycle
- test : **13,096행** — 100대지만 일찍 잘렸으므로 평균 131 cycle
- **정답은 100개**: test 엔진의 **마지막 cycle 한 시점만 채점**한다는 뜻 (중요)

---


## 3. EDA — 데이터 특성 파악  `[1-2 데이터 특성]`


In [ ]:
# --- 엔진 수명 분포: train vs test ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
mc_tr = train.groupby("unit")["cycle"].max()
mc_te = test.groupby("unit")["cycle"].max()
ax[0].hist(mc_tr, bins=30, color="#4CAF50", edgecolor="black")
ax[0].set_title(f"Train — 엔진 수명 (run-to-failure)  n={len(mc_tr)}")
ax[0].set_xlabel("cycles to failure"); ax[0].axvline(mc_tr.mean(), color="k", ls="--", label=f"평균 {mc_tr.mean():.0f}")
ax[0].legend()

ax[1].hist(mc_te, bins=30, color="#F44336", edgecolor="black")
ax[1].set_title(f"Test — 마지막 관측 cycle (truncation 시점)  n={len(mc_te)}")
ax[1].set_xlabel("cycles observed"); ax[1].axvline(mc_te.mean(), color="k", ls="--", label=f"평균 {mc_te.mean():.0f}")
ax[1].legend()
plt.suptitle("엔진 수명·관측구간 분포"); plt.tight_layout(); plt.show()

print(f"Train: min={mc_tr.min()} cycle, max={mc_tr.max()} cycle, 평균={mc_tr.mean():.0f}")
print(f"Test : min={mc_te.min()} cycle, max={mc_te.max()} cycle, 평균={mc_te.mean():.0f}")


**[관찰]** 같은 엔진 모델인데 수명이 **128~362 cycle** 로 천차만별 — 자동차로 치면 똑같이 만든 차도 어떤 건 10만km에, 어떤 건 30만km에 망가지는 셈.
test 엔진은 평균 131 cycle에서 잘려 있어 **다양한 RUL 시점** 을 다 잘 맞춰야 합니다.


In [ ]:
# --- 센서별 분산: 정보 없는 센서 찾기 ---
sensor_cols = [c for c in train.columns if c.startswith("sensor_")]
variances = train[sensor_cols].var().sort_values()
constant_sensors = variances[variances < 1e-4].index.tolist()   # 표준 drop 임계값

print("센서별 분산 (작은 순):")
print(variances.head(10))
print(f"\n분산 ≈ 0 인 상수 센서 ({len(constant_sensors)}개): {sorted(constant_sensors)}")


**[관찰]** 분산 0인 7개 센서(`sensor_1, 5, 6, 10, 16, 18, 19`)는 **시간이 지나도 안 변하는 정적 보정값**.
RUL과 무관하므로 **버립니다**: 21개 → 14개. 이게 Feature Selection의 첫 단계.


In [ ]:
# --- 1번 엔진의 14개 센서 시간 trend ---
informative = [c for c in sensor_cols if c not in constant_sensors]
engine1 = train[train["unit"] == 1]

fig, axes = plt.subplots(4, 4, figsize=(15, 10))
for i, col in enumerate(informative[:16]):
    ax = axes[i // 4, i % 4]
    ax.plot(engine1["cycle"], engine1[col], color="#1565C0", linewidth=0.8)
    ax.set_title(col, fontsize=9); ax.tick_params(labelsize=7)
for j in range(len(informative), 16):
    axes[j // 4, j % 4].axis("off")
plt.suptitle("Engine #1 — 사이클별 14개 센서 trend", fontsize=12, y=1.00)
plt.tight_layout(); plt.show()


**[관찰]** 두 가지 패턴이 보입니다:
- **단조 증가** (sensor_2, 3, 4, 11, 15, 17 등) — 시간 흐를수록 값이 올라감 = 열화 신호
- **단조 감소** (sensor_7, 12, 20, 21 등) — 시간 흐를수록 값이 내려감 = 효율 저하 신호

**🔑 핵심 직관**: 우리가 RUL을 예측할 수 있는 이유는 — 엔진이 죽기 전에 *"내가 점점 안 좋아지고 있어"* 라는
신호를 센서가 trend로 보내주기 때문입니다.

---
